In [1]:
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

df = pd.read_parquet("../data/processed/train_sample.parquet")
df.shape

(464457, 45)

In [2]:
summary = (
    df.dtypes.astype(str).to_frame("dtype")
    .join((df.isna().mean() * 100).round(2).rename("missing_pct"))
    .join(df.nunique().rename("n_unique"))
    .sort_values("missing_pct", ascending=False)
)
summary

,dtype,missing_pct,n_unique
ResourceType,str,99.93,17
ActionGrouped,str,99.40,3
ActionGranular,str,99.40,13
EmailClusterId,str,99.00,2573
ThreatFamily,str,98.90,333
AntispamDirection,str,98.14,5
Roles,str,98.11,8
SuspicionLevel,str,84.99,2
LastVerdict,str,76.64,4
MitreTechniques,str,56.90,459


In [3]:
KEY = ["OrgId", "IncidentId"]
per_incident = df.dropna(subset=["IncidentGrade"]).drop_duplicates(KEY)["IncidentGrade"]

pd.DataFrame(
    {
        "rows": df["IncidentGrade"].value_counts(normalize=True),
        "incidents": per_incident.value_counts(normalize=True),
    }
).round(4)

,rows,incidents
IncidentGrade,,
BenignPositive,0.4087,0.4911
FalsePositive,0.2167,0.3036
TruePositive,0.3746,0.2052


In [4]:
ts = pd.to_datetime(df["Timestamp"], utc=True, errors="coerce", format="ISO8601")
print("range:", ts.min(), "->", ts.max(), "| unparsed:", int(ts.isna().sum()))
ts.dt.date.value_counts().sort_index()

range: 2023-12-17 01:48:01+00:00 -> 2024-06-17 11:40:25+00:00 | unparsed: 0


Timestamp
2023-12-17        4
2023-12-29        4
2024-01-12        1
2024-01-21        4
2024-01-22        4
2024-01-23        1
2024-01-30        4
2024-02-17        4
2024-02-22        4
2024-02-29        1
2024-03-05        4
2024-03-06        4
2024-03-08        1
2024-03-26        1
2024-04-01        1
2024-04-03        4
2024-04-06        1
2024-04-07        1
2024-04-13        4
2024-04-14        4
2024-04-16        2
2024-04-23        1
2024-04-27        1
2024-05-02        3
2024-05-06        1
2024-05-07        1
2024-05-10        5
2024-05-14        5
2024-05-15        1
2024-05-20     3685
2024-05-21     3472
2024-05-22     2868
2024-05-23     1881
2024-05-24    12892
2024-05-25     6274
2024-05-26     6885
2024-05-27     8127
2024-05-28     1369
2024-05-29      955
2024-05-30     1074
2024-05-31     4456
2024-06-01     2520
2024-06-02     3453
2024-06-03    34588
2024-06-04    32226
2024-06-05    32170
2024-06-06    30050
2024-06-07    28000
2024-06-08    26845
2024-06-09

In [5]:
grouped = df.groupby(KEY)
size = grouped.agg(rows=("AlertId", "size"), alerts=("AlertId", "nunique"))
display(size.describe(percentiles=[0.5, 0.9, 0.99]).round(1))

first_label = grouped["IncidentGrade"].first()
print("incidents:", len(first_label), "| with no label:", int(first_label.isna().sum()))

,rows,alerts
count,36744.0,36744.0
mean,12.6,2.2
std,164.3,40.8
min,1.0,1.0
50%,4.0,1.0
90%,14.0,2.0
99%,100.0,9.0
max,10008.0,2975.0


incidents: 36744 | with no label: 1209


In [6]:
print("rows:", len(df), "| distinct Id:", df["Id"].nunique())
print("distinct (OrgId, AlertId):", df[["OrgId", "AlertId"]].drop_duplicates().shape[0])

rows: 464457 | distinct Id: 36744
distinct (OrgId, AlertId): 79941


In [7]:
from soc_triage.features.aggregate import build_incident_table
from soc_triage.models.split import summarize_splits, time_split
from soc_triage.utils.config import load_config

cfg = load_config()
entity_cols = [c for c in cfg["features"]["entity_columns"] if c in df.columns]
category_cols = [c for c in cfg["features"]["category_columns"] if c in df.columns]
print("entity:", entity_cols)
print("category:", category_cols)

table = build_incident_table(df, entity_cols, category_cols)
print(table.shape)
table.head()

entity: ['DeviceId', 'Sha256', 'IpAddress', 'Url', 'AccountSid', 'AccountUpn', 'AccountObjectId', 'AccountName', 'DeviceName', 'NetworkMessageId', 'RegistryKey', 'ApplicationId', 'FileName', 'FolderPath']
category: ['Category', 'EntityType', 'EvidenceRole', 'SuspicionLevel', 'LastVerdict', 'ResourceType', 'OSFamily']
(36744, 76)


,OrgId,IncidentId,n_evidence_rows,n_alerts,n_detectors,n_alert_titles,start_time,end_time,IncidentGrade,n_distinct_DeviceId,n_distinct_Sha256,n_distinct_IpAddress,n_distinct_Url,n_distinct_AccountSid,n_distinct_AccountUpn,n_distinct_AccountObjectId,n_distinct_AccountName,n_distinct_DeviceName,n_distinct_NetworkMessageId,n_distinct_RegistryKey,n_distinct_ApplicationId,n_distinct_FileName,n_distinct_FolderPath,n_mitre_techniques,Category__InitialAccess,Category__Exfiltration,Category__Impact,Category__SuspiciousActivity,Category__CommandAndControl,Category__Execution,...,EntityType__MailCluster,EntityType__CloudApplication,EvidenceRole__Related,EvidenceRole__Impacted,SuspicionLevel__Suspicious,SuspicionLevel__Incriminated,LastVerdict__Suspicious,LastVerdict__Malicious,LastVerdict__NoThreatsFound,LastVerdict__DomainPII_50d8b4a941c26b89482c94ab324b5a274f9ced66,ResourceType__Virtual_Machine,ResourceType__App_Service,ResourceType__Azure_Arc_machine,ResourceType__Key_Vault,ResourceType__Azure_Database_for_MySQL_Server,ResourceType__Subscription,ResourceType__Storage_Account,ResourceType__Key_Vault_Secret,ResourceType__SQL_Database,ResourceType__Azure_Resource,ResourceType__API_Management_Service,ResourceType__Azure_Database_for_PostgreSQL_Server,OSFamily__5,OSFamily__0,OSFamily__1,OSFamily__2,duration_min,start_hour,start_dow,start_is_weekend
0,809,58352,17,4,2,2,2024-06-13 04:52:55+00:00,2024-06-13 11:22:02+00:00,FalsePositive,4,2,4,3,2,2,2,2,5,1,1,1,2,2,2.0,0.294118,0.0,0.0,0.0,0.705882,0.0,...,0.00,0.176471,0.705882,0.294118,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.941176,0.058824,0.0,0.0,389.116667,4,3,0
1,6,2472,500,1,1,1,2024-05-26 21:07:09+00:00,2024-05-26 21:07:09+00:00,FalsePositive,1,1,500,1,1,1,1,1,1,1,1,1,1,1,0.0,0.000000,0.0,1.0,0.0,0.000000,0.0,...,0.00,0.000000,1.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.000000,0.000000,0.0,0.0,0.000000,21,6,1
2,68,125436,16,2,1,1,2024-06-11 18:52:33+00:00,2024-06-11 20:19:54+00:00,BenignPositive,1,1,2,1,2,2,2,2,1,2,1,1,1,1,1.0,1.000000,0.0,0.0,0.0,0.000000,0.0,...,0.25,0.000000,0.625000,0.375000,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.000000,0.000000,0.0,0.0,87.350000,18,1,0
3,0,213,9522,2707,6,6,2024-06-06 22:08:43+00:00,2024-06-09 05:32:19+00:00,TruePositive,1,1,516,1,1077,1077,1077,1077,1,1,1,2,1,1,5.0,0.994959,0.0,0.0,0.0,0.000000,0.0,...,0.00,0.000210,0.715501,0.284499,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.000000,0.000000,0.0,0.0,3323.600000,22,3,0
4,52,4549,273,1,1,1,2024-06-09 21:06:11+00:00,2024-06-09 21:06:11+00:00,BenignPositive,1,1,51,224,1,1,1,1,1,1,1,1,1,1,0.0,0.000000,0.0,0.0,0.0,1.000000,0.0,...,0.00,0.000000,1.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.000000,0.000000,0.0,0.0,0.000000,21,6,1


In [8]:
train, valid, test = time_split(table, cfg["split"]["train_frac"], cfg["split"]["valid_frac"])
summarize_splits(train, valid, test)

[2026-11-14 13:46:36,498] WARNING soc_triage.models.split:17 - Dropped 1209 incidents with no label or no start time


,split,incidents,start,end,BenignPositive,FalsePositive,TruePositive
0,train,24874,2023-12-17 01:48:01+00:00,2024-06-12 04:36:08+00:00,0.5118,0.2731,0.2151
1,valid,5331,2024-06-12 04:36:08+00:00,2024-06-13 17:01:54+00:00,0.4127,0.4296,0.1578
2,test,5330,2024-06-13 17:02:18+00:00,2024-06-17 11:40:25+00:00,0.4734,0.3203,0.2064


In [9]:
labelled = table.dropna(subset=["IncidentGrade"])
cols = ["n_evidence_rows", "n_alerts", "n_detectors", "n_mitre_techniques", "duration_min"]
labelled.groupby("IncidentGrade")[cols].median()

,n_evidence_rows,n_alerts,n_detectors,n_mitre_techniques,duration_min
IncidentGrade,,,,,
BenignPositive,5.0,1.0,1.0,0.0,0.0
FalsePositive,4.0,1.0,1.0,0.0,0.0
TruePositive,3.0,1.0,1.0,0.0,0.0
